In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NPOC"] = "1"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".5"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["NPROC"] = "1"
os.environ["intra_op_parallelism_threads"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from gravpop import *
import h5py
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

### For saving and loading
### should be in newest gravpop version
_NONE_ATTR = "__is_none__"

def save_dict_h5(filename, data):
    def _save_group(h5group, d):
        for k, v in d.items():
            if v is None:
                g = h5group.create_group(k)
                g.attrs[_NONE_ATTR] = True
            elif isinstance(v, dict):
                _save_group(h5group.create_group(k), v)
            elif isinstance(v, pd.DataFrame):
                g = h5group.create_group(k)
                g.create_dataset("columns", data=np.array(v.columns, dtype="S"))
                g.create_dataset("values", data=v.to_numpy())
            elif isinstance(v, (list, tuple)):
                arr = np.array(v)
                if arr.dtype.kind in {"U", "O"}:  # strings (or objects that are strings)
                    arr = arr.astype("S")
                h5group.create_dataset(k, data=arr)
            elif isinstance(v, str):
                h5group.create_dataset(k, data=np.bytes_(v))  # NumPy 2.0+
            else:
                h5group.create_dataset(k, data=np.array(v))
    with h5py.File(filename, "w") as f:
        _save_group(f, data)

def load_dict_h5(filename):
    def _load_group(h5group):
        out = {}
        for k, v in h5group.items():
            if isinstance(v, h5py.Group):
                # None sentinel?
                if v.attrs.get(_NONE_ATTR, False):
                    out[k] = None
                # DataFrame?
                elif "columns" in v and "values" in v:
                    cols = [c.decode() for c in v["columns"][()]]
                    out[k] = pd.DataFrame(v["values"][()], columns=cols)
                else:
                    out[k] = _load_group(v)
            else:
                arr = v[()]
                if arr.dtype.kind == "S":
                    # Return string arrays as Python lists of str; scalars as str
                    out[k] = arr.decode() if arr.ndim == 0 else [x.decode() for x in arr.flatten()]
                else:
                    out[k] = arr.tolist() if arr.ndim == 0 else arr
        return out
    with h5py.File(filename, "r") as f:
        return _load_group(f)

KeyboardInterrupt: 

# Load Data

In [2]:
event_data = load_dict_h5("/home/asad.hussain/O4b_test/o4a_data_products/event_data.hdf5");
selection_data = load_dict_h5("/home/asad.hussain/O4b_test/o4a_data_products/selection_data.h5");
analysis_time = selection_data.pop('analysis_time')
total_generated = selection_data.pop('total_generated')
total_detected = selection_data.pop('total_detected')
selection_func = SelectionFunction(selection_data, analysis_time=analysis_time, total_generated=total_generated, total_detected=total_detected);

In [3]:
for k, v in selection_data.items():
    if np.any(~np.isfinite(v)):
        print(k)

In [4]:
for k, v in event_data.items():
    if np.any(~np.isfinite(v)):
        print(k)

# Define Models

In [5]:
smoothed_mass_model = SmoothedTwoComponentPrimaryMassRatio(mmin_fixed=2, mmax_fixed=300, gaussian_mass_maximum=100,
                                 var_names=['mass_1_source', 'mass_ratio'],
                                 hyper_var_names=['alpha', 'beta', 'lam', 'mpp', 'sigpp', 'delta_m', 'mmin', 'mmax'])
    

redshift_model = PowerLawRedshift(var_names=['redshift'],hyper_var_names=['lamb'], z_max=3);

"""
the_spin_orientation_model_2D = GaussianIsotropicSpinOrientationsIIDAnalytic2D(
    a=-1, b=1,
    var_names=['cos_tilt_1', 'cos_tilt_2'],
    hyper_var_names=['xi_spin','sigma_spin']
)

the_spin_orientation_model_1D = GaussianIsotropicSpinOrientationsIIDAnalytic(
    a=-1, b=1,
    var_names=['cos_tilt_1', 'cos_tilt_2'],
    hyper_var_names=['xi_spin','sigma_spin']
)
"""

model_tilt1__ = TruncatedGaussian2DAnalytic(a = [-1,-1], b = [1,1],
                                 var_names=['cos_tilt_1', 'cos_tilt_2'],
                                 hyper_var_names=['mu_spin', 'sigma_spin', 'mu_spin', 'sigma_spin', 'rho_cos_tilt'])

model_tilt1 = FixedParameters(model_tilt1__, {'rho_cos_tilt' : 1e-4}) ## Dont do zero because derivatives can be nan


model_tilt_uniform = Uniform2DAnalytic(a=[-1,-1], b=[1,1],
                      var_names=['cos_tilt_1', 'cos_tilt_2'],
                      hyper_var_names=[])

model_tilt = mixture([model_tilt1, model_tilt_uniform], ['xi_spin', 'one_minus_xi_spin'])


chi_model = TruncatedGaussian2DAnalytic(a = [0,0], b = [1,1],
                                 var_names=['chi_1', 'chi_2'],
                                 hyper_var_names=['mu_chi', 'sigma_chi', 'mu_chi', 'sigma_chi', 'rho_chi'])

chi_model = FixedParameters(chi_model, {'rho_chi' : 1e-4})

"""
var_names = ['chi_1', 'chi_2'];
a = [0,0]; b=[1,1];

spin_2D_a = TruncatedGaussian2DAnalytic(
    var_names=var_names,
    hyper_var_names=['mu_chi_1_at_0', 'sigma_chi_1_at_0', 'mu_chi_2_at_0', 'sigma_chi_2_at_0', 'rho_chi_1'],
    a=a,
    b=b
)
spin_2D_b = TruncatedGaussian2DAnalytic(
    var_names=var_names,
    hyper_var_names=['mu_chi_1', 'sigma_chi_1', 'mu_chi_2', 'sigma_chi_2', 'rho_chi_2'],
    a=a,
    b=b
)

full_spin_model = mixture(
    [spin_2D_a, spin_2D_b],
    ['eta_spin', 'one_minus_eta_spin']
)
"""

"\nvar_names = ['chi_1', 'chi_2'];\na = [0,0]; b=[1,1];\n\nspin_2D_a = TruncatedGaussian2DAnalytic(\n    var_names=var_names,\n    hyper_var_names=['mu_chi_1_at_0', 'sigma_chi_1_at_0', 'mu_chi_2_at_0', 'sigma_chi_2_at_0', 'rho_chi_1'],\n    a=a,\n    b=b\n)\nspin_2D_b = TruncatedGaussian2DAnalytic(\n    var_names=var_names,\n    hyper_var_names=['mu_chi_1', 'sigma_chi_1', 'mu_chi_2', 'sigma_chi_2', 'rho_chi_2'],\n    a=a,\n    b=b\n)\n\nfull_spin_model = mixture(\n    [spin_2D_a, spin_2D_b],\n    ['eta_spin', 'one_minus_eta_spin']\n)\n"

# Define Priors

In [6]:
mass_redshift_priors = dict(
    alpha 			= dist.Uniform(-4,12),
    lam 			= dist.Uniform(0,1),
    mmin 			= dist.Uniform(2,6),
    mmax 			= DiracDelta(300),
    beta 			= dist.Uniform(-2,7),
    mpp 			= dist.Uniform(20,50),
    sigpp 			= dist.Uniform(1,10),
    delta_m 		= dist.Uniform(0,12),
    lamb 			= dist.Uniform(-10,10)
)

spin_mag_priors = dict(
    mu_chi = dist.Uniform(0.1, 0.9),
    sigma_chi = dist.Uniform(0.1, 1), # NOTE: changed from 0 min bbound
)

spin_orientation_standard_priors = dict(
    mu_spin = dist.Uniform(-1, 1),
    xi_spin 	= dist.Uniform(0,1),
    sigma_spin 	= dist.Uniform(0.2, 1)
)

priors = mass_redshift_priors.copy()
priors.update(spin_mag_priors)
priors.update(spin_orientation_standard_priors)

# Define Likelihood

In [7]:
HL = MarginalizedHybridLikelihood(
    event_data=event_data,
    selection_data=selection_func,
    models=[smoothed_mass_model, redshift_model, model_tilt, chi_model],
    models_selection=[smoothed_mass_model, redshift_model, model_tilt, chi_model],
    fix_kernels_selection={},
    fix_kernels_events={}
)

In [8]:
rkeys = jax.random.split(jax.random.key(1), len(priors))
test = {
    k : v.value if isinstance(v, DiracDelta) else v.sample(rk)
    for ((k, v), rk) in zip(priors.items(), rkeys)
}

In [9]:
test

{'alpha': Array(7.241844, dtype=float32),
 'lam': Array(0.40364635, dtype=float32),
 'mmin': Array(4.2311363, dtype=float32),
 'mmax': 300,
 'beta': Array(4.0600433, dtype=float32),
 'mpp': Array(21.385798, dtype=float32),
 'sigpp': Array(8.135462, dtype=float32),
 'delta_m': Array(9.449367, dtype=float32),
 'lamb': Array(4.02849, dtype=float32),
 'mu_chi': Array(0.58264035, dtype=float32),
 'sigma_chi': Array(0.9750444, dtype=float32),
 'mu_spin': Array(0.46869493, dtype=float32),
 'xi_spin': Array(0.9177871, dtype=float32),
 'sigma_spin': Array(0.7873723, dtype=float32)}

In [10]:
HL.logpdf(test)

Array(-1340.8604, dtype=float32)

# Define Sampler

In [ ]:
samp = Sampler(
    priors=priors,
    latex_symbols={k:k for k in priors.keys()},
    likelihood=HL,
    validation=True
);

: 

# Run Sampler

In [ ]:
samp.sample()

Validation enabled


F0211 11:12:27.717440 1958491 env.cc:84] Check failed: ret == 0 (11 vs. 0) Thread tf_foreach creation via pthread_create() failed.


In [ ]:
samp.samples.to_csv("result_o4a_old_mass_model.csv")